# OpenDistillation v0 Demo

> A personal model factory for the AI PC and AI phone era.

This notebook is the first runnable prototype for one narrow model type: a notes / school model from TXT/MD notes. It covers text upload/loading, validation, chunking, dataset schema validation, deterministic mock teacher generation by default, optional local Qwen teacher generation, a deterministic dataset quality report, an optional short student fine-tuning entry point, and an optional multi-question before/after quality report.

Safe default run order: setup -> dependency install skipped -> sample notes -> chunks -> mock teacher rows -> JSONL preview -> dataset quality report -> training skipped -> model quality report skipped -> export placeholder.

Optional GPU run order: choose a Colab GPU runtime -> set `INSTALL_TRAINING_DEPS = True` -> rerun setup if Colab restarts -> optionally set `RUN_REAL_TEACHER = True` -> optionally set `RUN_TRAINING = True` -> run the bounded before/after quality report.

By default it does not train a model, download a model, use a GPU, call paid APIs, or export GGUF files. The real teacher and training cells stay skipped until you explicitly opt in from a Colab GPU runtime.

Dataset quality checks run locally and deterministically. Model quality remains a bounded qualitative smoke report, not a benchmark.

## Step 1: Runtime setup

Run this notebook from the repository root locally. In Colab, opening the notebook from GitHub starts in `/content`, so the setup cell clones the OpenDistillation repository before importing local helpers. The default path uses only Python standard-library code and local helper modules. Optional real teacher generation and optional training require extra Hugging Face packages and a GPU.

The setup cell also creates a runtime status log. If Colab's output pane fails, open the Colab Terminal and run `cat /tmp/opendistillation_status.jsonl` to recover the latest `OD_STATUS` markers.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

OPEN_DISTILLATION_REPO_URL = "https://github.com/tacotuesday8888/OpenDistillation.git"
STATUS_LOG_PATH = Path(tempfile.gettempdir()) / "opendistillation_status.jsonl"


def is_colab_runtime():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _status_safe_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return list(value)
    return value


def record_status(stage, status, **details):
    record = {"stage": stage, "status": status}
    for key, value in details.items():
        record[key] = _status_safe_value(value)

    STATUS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with STATUS_LOG_PATH.open("a", encoding="utf-8") as status_file:
        status_file.write(json.dumps(record, ensure_ascii=False) + "\n")

    detail_text = ""
    if details:
        detail_text = " details=" + json.dumps(record, ensure_ascii=False)
    print(f"OD_STATUS stage={stage} status={status}{detail_text}", flush=True)


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "opendistillation").exists() and (PROJECT_ROOT.parent / "src" / "opendistillation").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "opendistillation").exists():
    if is_colab_runtime():
        PROJECT_ROOT = Path("/content/OpenDistillation")
        if not (PROJECT_ROOT / "src" / "opendistillation").exists():
            subprocess.check_call(["git", "clone", "--depth", "1", OPEN_DISTILLATION_REPO_URL, str(PROJECT_ROOT)])
    else:
        raise RuntimeError("OpenDistillation source files were not found. Run this notebook from the repository root, or open the GitHub notebook in Colab.")

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from opendistillation import (
    BeforeAfterComparisonEngine,
    DEFAULT_REAL_TEACHER_MODEL,
    HuggingFaceLocalTeacherEngine,
    MockTeacherEngine,
    OPTIONAL_TRAINING_INSTALL_PACKAGES,
    OPTIONAL_TRAINING_PACKAGES,
    TeacherRequest,
    SFTLoRAConfig,
    SFTLoRATrainingEngine,
    analyze_dataset_quality,
    build_comparison_request,
    build_pip_install_command,
    build_training_request,
    check_training_runtime,
    chunk_text,
    explain_runtime_failure,
    explain_teacher_failure,
    format_dataset_quality_report,
    format_runtime_check,
    load_text_document,
    rows_to_jsonl,
)

STATUS_LOG_PATH.write_text("", encoding="utf-8")
record_status("setup", "ready", project_root=PROJECT_ROOT, colab=is_colab_runtime())

print(f"Using project root: {PROJECT_ROOT}")
print(f"Status log: {STATUS_LOG_PATH}")
print("If Colab output display fails, open Terminal and run: cat /tmp/opendistillation_status.jsonl")
print("Runtime: mock/data-prep path runs on CPU. Optional training is skipped unless RUN_TRAINING is set to True later.")
print("Smoke-test note: tiny optional runs prove wiring, not model quality.")


## Step 2: Optional dependency install

Keep `INSTALL_TRAINING_DEPS = False` for the default local demo path. In Colab, switch to a GPU runtime first, then set this to `True` once to install the optional Hugging Face teacher/training packages. The install command intentionally does not upgrade Colab's preinstalled GPU `torch` package; the runtime check still verifies that `torch` and CUDA are available before training starts.

This cell writes `OD_STATUS` markers for configured, started, skipped, succeeded, or failed install states.


In [ ]:
INSTALL_TRAINING_DEPS = False

install_command = build_pip_install_command()
record_status(
    "install",
    "configured",
    install_training_deps=INSTALL_TRAINING_DEPS,
    packages=list(OPTIONAL_TRAINING_INSTALL_PACKAGES),
    command=install_command,
)

print("Runtime packages checked before training:")
print(", ".join(OPTIONAL_TRAINING_PACKAGES))
print("Packages installed by this cell:")
print(", ".join(OPTIONAL_TRAINING_INSTALL_PACKAGES))
print("Install command:")
print(install_command)

if INSTALL_TRAINING_DEPS:
    import subprocess

    record_status("install", "started")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *OPTIONAL_TRAINING_INSTALL_PACKAGES])
    except Exception as exc:
        record_status("install", "failed", error_type=type(exc).__name__, error=str(exc))
        raise
    record_status("install", "succeeded")
    print("Optional Hugging Face dependencies installed. Restart the runtime if Colab asks, then rerun setup.")
else:
    record_status("install", "skipped")
    print("Install skipped. Keep this false for the local CPU demo path.")


## Step 3: Upload or load a TXT/MD file

By default this cell uses `examples/sample-notes.md` so the notebook can run top to bottom in Colab or locally without a file picker. Set `USE_SAMPLE_NOTES = False` to upload one `.txt` or `.md` notes file instead.


In [ ]:
USE_SAMPLE_NOTES = True


def load_uploaded_or_sample():
    sample_path = PROJECT_ROOT / "examples" / "sample-notes.md"
    if USE_SAMPLE_NOTES:
        return sample_path.name, sample_path.read_text(encoding="utf-8")

    try:
        from google.colab import files  # type: ignore
    except ImportError:
        raise ValueError("Set USE_SAMPLE_NOTES = True locally, or run in Colab to upload one .txt or .md file.")

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded. Upload one .txt or .md file to continue.")
    filename, content = next(iter(uploaded.items()))
    return filename, content


filename, content = load_uploaded_or_sample()
document = load_text_document(filename, content)

print(f"File: {document.filename}")
print(f"Extension: {document.extension}")
print(f"Characters: {document.char_count}")
print(f"Approx. words: {document.word_count}")
if document.warnings:
    print("Warnings:")
    for warning in document.warnings:
        print(f"- {warning}")
print("\nPreview:\n")
print(document.preview)


## Step 4: Chunk the document

The v0 chunker prefers paragraph boundaries, preserves source order, removes empty chunks, and assigns stable IDs like `chunk-0001`. The first public demo should use a small notes file; larger files and many more rows are later hardening work.


In [ ]:
chunks = chunk_text(document.text, max_chars=300)
print(f"Chunks: {len(chunks)}")

for chunk in chunks[:3]:
    print("=" * 72)
    print(f"{chunk.id} | chars={chunk.char_count} | words={chunk.word_count}")
    print(chunk.text[:500])

## Step 5: Generate mock training examples by default

This prototype uses `MockTeacherEngine`, a deterministic local teacher path, by default. Set `RUN_REAL_TEACHER = True` only in a Colab GPU runtime after installing optional Hugging Face packages to use `Qwen/Qwen2.5-1.5B-Instruct` as a local open-source teacher. Both paths keep the same JSONL schema.

This cell writes `OD_STATUS` markers for teacher configuration, model-generation start, success, or failure. The real-teacher smoke test proves the teacher/training wiring can run; it is not a model-quality benchmark.


In [ ]:
RUN_REAL_TEACHER = False

teacher_request = TeacherRequest(chunks=chunks, examples_per_chunk=4)
teacher_engine_name = "huggingface-local-teacher" if RUN_REAL_TEACHER else "mock-local-teacher"
record_status(
    "teacher",
    "configured",
    run_real_teacher=RUN_REAL_TEACHER,
    engine=teacher_engine_name,
    chunk_count=len(chunks),
    examples_per_chunk=teacher_request.examples_per_chunk,
)

try:
    if RUN_REAL_TEACHER:
        teacher_engine = HuggingFaceLocalTeacherEngine()
        record_status("teacher", "real_started", model=DEFAULT_REAL_TEACHER_MODEL)
        print(f"Real teacher model: {DEFAULT_REAL_TEACHER_MODEL}")
        rows = teacher_engine.generate(teacher_request)
    else:
        teacher_engine = MockTeacherEngine()
        record_status("teacher", "mock_started", engine=teacher_engine.name)
        rows = teacher_engine.generate(teacher_request)

    dataset_jsonl = rows_to_jsonl(rows)
except Exception as exc:
    record_status("teacher", "failed", engine=teacher_engine_name, error_type=type(exc).__name__, error=str(exc))
    if RUN_REAL_TEACHER:
        print("Real teacher generation failed with a recoverable setup/runtime issue.")
        for line in explain_teacher_failure(exc):
            print(f"- {line}")
    raise

record_status(
    "teacher",
    "succeeded",
    engine=teacher_engine.name,
    generated_examples=len(rows),
    sends_data_remote=teacher_engine.sends_data_remote,
)

print(f"Teacher engine: {teacher_engine.name}")
print(f"Sends text to remote endpoint: {teacher_engine.sends_data_remote}")
print(f"Generated examples: {len(rows)}")
print("Smoke-test note: generated rows prove the flow is wired, not that model quality is good.")
print("\nFirst 5 examples:\n")
for row in rows[:5]:
    print(json.dumps(row, ensure_ascii=False, indent=2))


## Step 6: Dataset JSONL preview and dataset quality report

The initial schema is one JSON object per line with exactly these fields: `instruction`, `response`, and `source_chunk_id`. The dataset is saved to the runtime temp directory, not the repository.

This step checks dataset quality, not model quality. It looks for basic problems a beginner can understand before training: row count, chunk coverage, duplicate or near-duplicate questions, short or long answers, missing fields, and unexpected source chunk IDs.

In [ ]:
print("First JSONL lines:\n")
print("\n".join(dataset_jsonl.splitlines()[:5]))

print("\n" + "=" * 72)
dataset_quality_report = analyze_dataset_quality(
    rows,
    expected_chunk_ids=[chunk.id for chunk in chunks],
)
record_status(
    "dataset_quality",
    "reported",
    rows=dataset_quality_report.row_count,
    valid_rows=dataset_quality_report.valid_row_count,
    covered_chunks=len(dataset_quality_report.covered_chunk_ids),
    expected_chunks=len(dataset_quality_report.expected_chunk_ids),
    issues=len(dataset_quality_report.issues),
)
for line in format_dataset_quality_report(dataset_quality_report):
    print(line)
print("Dataset quality checks the generated rows. It does not prove the trained model is useful yet.")

# Save to the runtime temp directory, not the repository, so generated data is not committed.
output_path = Path(tempfile.gettempdir()) / "opendistillation_training_data.jsonl"
output_path.write_text(dataset_jsonl, encoding="utf-8")
record_status("dataset", "saved", rows=len(rows), output_path=output_path)
print(f"Saved runtime dataset to {output_path}")
print(f"Status log: {STATUS_LOG_PATH}")

try:
    from google.colab import files  # type: ignore
except ImportError:
    print("Download helper is available only in Colab; use the path above locally.")
else:
    files.download(str(output_path))

## Step 7: Optional short student fine-tuning

This is the first real training entry point, but it is off by default. It uses one small student model, `Qwen/Qwen2.5-0.5B-Instruct`, with TRL `SFTTrainer` and PEFT LoRA.

Leave `RUN_TRAINING = False` when you want the notebook to run without GPU or model downloads. Set it to `True` only in a Colab GPU runtime after installing the optional Hugging Face training packages and confirming the runtime check sees `torch`, CUDA, `transformers`, `datasets`, `trl`, `peft`, and `accelerate`. Output goes under `outputs/`, which is ignored by git.

This cell writes `OD_STATUS` markers for training configuration, runtime check, start, success, or failure. A 1-step smoke run proves wiring and artifact creation, not model quality.


In [ ]:
RUN_TRAINING = False

training_config = SFTLoRAConfig()
training_request = build_training_request(
    rows,
    output_dir=PROJECT_ROOT / "outputs" / "notes-lora",
    config=training_config,
)
training_engine = SFTLoRATrainingEngine(training_config)
training_plan = training_engine.describe(training_request)

record_status(
    "training",
    "configured",
    run_training=RUN_TRAINING,
    output_dir=training_request.output_dir,
    max_steps=training_request.max_steps,
    student_model=training_request.student_model,
)

print("Training plan:")
for key, value in training_plan.items():
    print(f"- {key}: {value}")
print("Smoke-test note: short adapter runs prove wiring, not model quality.")

training_result = None
if RUN_TRAINING:
    record_status("training", "runtime_check_started")
    runtime_check = check_training_runtime()
    record_status(
        "training",
        "runtime_check_finished",
        can_run_training=runtime_check.can_run_training,
        cuda_available=runtime_check.cuda_available,
        gpu_name=runtime_check.gpu_name,
        missing_packages=list(runtime_check.missing_packages),
        import_errors=runtime_check.import_errors,
    )
    print("Runtime check:")
    for line in format_runtime_check(runtime_check):
        print(f"- {line}")

    if not runtime_check.can_run_training:
        record_status("training", "blocked", reason="runtime_not_ready")
        raise RuntimeError("Training runtime is not ready. Install missing packages or switch to a GPU runtime, then rerun this cell.")

    try:
        record_status("training", "started", student_model=training_request.student_model, max_steps=training_request.max_steps)
        training_result = training_engine.train(training_request)
    except Exception as exc:
        record_status("training", "failed", error_type=type(exc).__name__, error=str(exc))
        print("Training failed with a recoverable setup/runtime issue.")
        for line in explain_runtime_failure(exc):
            print(f"- {line}")
        raise

    record_status(
        "training",
        "succeeded",
        engine=training_result.engine_name,
        adapter_output=training_result.output_path,
        created_model_artifact=training_result.created_model_artifact,
    )
    print(f"Training engine: {training_result.engine_name}")
    print(f"Adapter output: {training_result.output_path}")
    for note in training_result.notes:
        print(f"- {note}")
else:
    record_status("training", "skipped")
    print("Training skipped. Set RUN_TRAINING = True in a Colab GPU runtime to start the short SFT run.")


## Step 8: Before/after model quality report

This comparison uses up to three generated dataset questions. It runs only after the optional training cell creates a LoRA adapter. The result is a bounded qualitative quality report, not a benchmark.

The report separates reference answers from base-model and trained-adapter answers. It also prints a crude reference-overlap signal so a beginner can see whether the adapter answer is moving toward the note-grounded teacher answer. Read the answers before trusting the score.

This cell writes `OD_STATUS` markers for comparison skipped, configured, started, succeeded, or failed states.

In [ ]:
COMPARISON_MAX_EXAMPLES = 3

if training_result is None:
    record_status("comparison", "skipped", reason="training_result_is_none")
    print("Model quality report skipped because training did not run.")
    print("The dataset quality report above is still useful; it checks the generated rows before any model training.")
else:
    comparison_request = build_comparison_request(
        rows,
        training_result,
        config=training_config,
        max_examples=COMPARISON_MAX_EXAMPLES,
    )
    comparison_engine = BeforeAfterComparisonEngine()
    comparison_plan = comparison_engine.describe(comparison_request)
    record_status(
        "comparison",
        "configured",
        question_count=len(comparison_request.examples),
        questions=[example.question for example in comparison_request.examples],
        adapter_path=comparison_request.adapter_path,
        max_new_tokens=comparison_request.max_new_tokens,
    )
    print("Comparison plan:")
    for key, value in comparison_plan.items():
        print(f"- {key}: {value}")

    try:
        record_status("comparison", "started", engine=comparison_engine.name)
        comparison_result = comparison_engine.compare(comparison_request)
        record_status(
            "comparison",
            "succeeded",
            engine=comparison_engine.name,
            question_count=len(comparison_result.items),
        )
        print("\nModel quality report")
        for note in comparison_result.notes:
            print(f"- {note}")
        for index, item in enumerate(comparison_result.items, start=1):
            print("\n" + "=" * 72)
            print(f"Question {index}/{len(comparison_result.items)} from {item.source_chunk_id}")
            print(f"Question: {item.question}")
            print(f"Reference answer: {item.reference_response}")
            print(f"Base answer ({item.base_reference_overlap:.3f} reference overlap): {item.base_answer}")
            print(
                "Trained adapter answer "
                f"({item.trained_reference_overlap:.3f} reference overlap, "
                f"delta {item.overlap_delta:+.3f}): {item.trained_answer}"
            )
    except Exception as exc:
        record_status("comparison", "failed", error_type=type(exc).__name__, error=str(exc))
        print("Before/after comparison failed.")
        for line in explain_runtime_failure(exc):
            print(f"- {line}")
        raise

## Manual Colab smoke-test checklist

Use this checklist before marking the GPU path verified: GPU runtime selected, optional dependency install succeeds, real teacher download/load status is recorded when `RUN_REAL_TEACHER = True`, runtime check prints a GPU name before training, model download starts, short training starts, adapter output prints under `outputs/notes-lora/adapter`, before/after comparison prints both answers, total runtime is recorded, and any memory failure is recorded with the exact error.

If the Colab output pane fails, use the Terminal command printed by the setup cell to inspect `/tmp/opendistillation_status.jsonl`. The status markers are evidence of execution state, not a quality benchmark.


## Export placeholder

GGUF export and local runtime instructions are later milestones. The intended `ExportEngine` plug-in point is after training output exists: convert or merge the model output, then document llama.cpp and/or Ollama-style local commands.

In [ ]:
print("Export placeholder: skipped.")
print("No GGUF files, model artifacts, or local runtime files are created by this prototype.")